# TUM-VIE: Kalman-Style vs Recurrent Fusion

Quick model/API comparison for `KalmanStyleSpikingFusionSurrogate` and `VisualIMURecurrentFusionBlock` using Tonic `TUMVIE`.

In [ ]:
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
from tonic.datasets import TUMVIE
from tonic import transforms
import spyx.models as fm

SEED = 42
N_BINS = 20
SPATIAL_FACTOR = 0.2
data_root = Path('../data').resolve()
sensor = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR), int(sensor[1] * SPATIAL_FACTOR), sensor[2])
tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_BINS),
])
ds = TUMVIE(save_to=str(data_root), recording='mocap-6dof', transform=tfm)
d, t = ds[0]
x = np.concatenate([np.asarray(d['events_left']), np.asarray(d['events_right'])], axis=1).astype(np.float32)
imu = np.asarray(d['imu'], dtype=np.float32)
imu = imu.mean(axis=0) if imu.ndim == 2 and imu.shape[0] > 0 else np.zeros((6,), dtype=np.float32)
x_batch = jnp.asarray(np.stack([x] * 4, axis=0))
u_batch = jnp.asarray(np.stack([imu] * 4, axis=0))
print('x_batch:', x_batch.shape, 'u_batch:', u_batch.shape)

In [ ]:
def recurrent_forward(xb, ub):
    cfg = fm.VisualIMURecurrentConfig(
        vision_cfg=fm.ConvConfig(T=int(xb.shape[1]), in_channels=int(xb.shape[2]), hidden_channels=(24, 48), output_dim=64, dt=1.0),
        imu_dim=int(ub.shape[-1]), hidden_dim=96, output_dim=3,
    )
    return fm.VisualIMURecurrentFusionBlock(cfg)(xb, ub)

def kalman_forward(xb, ub):
    cfg = fm.KalmanFusionConfig(
        vision_cfg=fm.ConvConfig(T=int(xb.shape[1]), in_channels=int(xb.shape[2]), hidden_channels=(24, 48), output_dim=64, dt=1.0),
        imu_dim=int(ub.shape[-1]), state_dim=32, output_dim=3,
    )
    return fm.KalmanStyleSpikingFusionSurrogate(cfg)(xb, ub)

recurrent = hk.without_apply_rng(hk.transform(lambda xb, ub: recurrent_forward(xb, ub)))
kalman = hk.without_apply_rng(hk.transform(lambda xb, ub: kalman_forward(xb, ub)))

pr = recurrent.init(jax.random.PRNGKey(SEED), x_batch, u_batch)
pk = kalman.init(jax.random.PRNGKey(SEED + 1), x_batch, u_batch)

yr = recurrent.apply(pr, x_batch, u_batch)
yk = kalman.apply(pk, x_batch, u_batch)
print('recurrent output:', yr.shape)
print('kalman output   :', yk.shape)

Next: add shared train/eval loops, tune state/hidden dims, and compare MAE on held-out recordings (`running-easy`, `bike-easy`).